# 01 · Data Loading & First Look

**Goal:** load the raw Olist CSV files, understand their size and structure, and check data quality
before cleaning.

**Source:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) —
~100k orders placed on Brazilian marketplaces between 2016 and 2018.

We use 7 of the 9 tables. `geolocation` and `sellers` are not needed for the business questions of this project.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW = Path("../data/raw")

## 1. Load the tables

In [2]:
FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

dfs = {name: pd.read_csv(RAW / file) for name, file in FILES.items()}

overview = pd.DataFrame(
    {
        "rows": {n: len(df) for n, df in dfs.items()},
        "columns": {n: df.shape[1] for n, df in dfs.items()},
        "memory_mb": {n: df.memory_usage(deep=True).sum() / 1e6 for n, df in dfs.items()},
    }
)
overview

,rows,columns,memory_mb
orders,99441,8,55.51
order_items,112650,7,37.74
order_payments,103886,5,17.02
order_reviews,99224,7,41.03
customers,99441,5,27.88
products,32951,9,6.60
category_translation,71,2,0.01


## 2. Columns and data types

In [3]:
for name, df in dfs.items():
    print(f"--- {name} ---")
    print(df.dtypes.to_string(), "\n")

--- orders ---
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str 

--- order_items ---
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64 

--- order_payments ---
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64 

--- order_reviews ---
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str 

--- cust

Date columns are loaded as `object` (strings). They will be converted to `datetime` in the cleaning step.

## 3. Missing values

In [4]:
missing = []
for name, df in dfs.items():
    na = df.isna().sum()
    for col, n in na[na > 0].items():
        missing.append({"table": name, "column": col, "missing": n, "share_%": n / len(df) * 100})

pd.DataFrame(missing).sort_values("share_%", ascending=False).reset_index(drop=True)

,table,column,missing,share_%
0,order_reviews,review_comment_title,87656,88.34
1,order_reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_name_lenght,610,1.85
4,products,product_category_name,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,orders,order_delivered_carrier_date,1783,1.79
8,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


**Observations:**
- Review title and message are empty for most reviews — expected, writing a comment is optional. We only need `review_score`.
- Missing delivery dates belong to orders that were never delivered (canceled, unavailable, in transit).
- ~600 products have no category; they will be labelled `unknown`.

## 4. Duplicates and primary keys

In [5]:
KEYS = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "category_translation": ["product_category_name"],
}

pd.DataFrame(
    {
        "full_row_duplicates": {n: dfs[n].duplicated().sum() for n in KEYS},
        "key_duplicates": {n: dfs[n].duplicated(subset=k).sum() for n, k in KEYS.items()},
    }
)

,full_row_duplicates,key_duplicates
orders,0,0
order_items,0,0
order_payments,0,0
order_reviews,0,0
customers,0,0
products,0,0
category_translation,0,0


In [6]:
reviews = dfs["order_reviews"]
print("Orders with more than one review:", (reviews["order_id"].value_counts() > 1).sum())
print("review_id reused across orders:  ", (reviews["review_id"].value_counts() > 1).sum())

Orders with more than one review: 547
review_id reused across orders:   789


The key combinations are unique, but **547 orders have several reviews**, and **789 `review_id`s are reused across orders** — so `review_id` alone is not a valid primary key. For analysis we keep **one review per order** (the latest one), and `order_id` becomes the key of the clean reviews table.

## 5. Customer identifiers

In [7]:
customers = dfs["customers"]
print(f"customer_id:        {customers['customer_id'].nunique():,}")
print(f"customer_unique_id: {customers['customer_unique_id'].nunique():,}")

customer_id:        99,441
customer_unique_id: 96,096


`customer_id` is created for **every order**, so it cannot detect repeat buyers.
The real person is `customer_unique_id` — we use it for retention, cohorts and RFM.

## 6. Order status and period

In [8]:
orders = dfs["orders"]
status = orders["order_status"].value_counts().to_frame("orders")
status["share_%"] = status["orders"] / len(orders) * 100
status

,orders,share_%
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


In [9]:
purchase = pd.to_datetime(orders["order_purchase_timestamp"])
print("First order:", purchase.min())
print("Last order: ", purchase.max())
purchase.dt.to_period("M").value_counts().sort_index().tail(8)

First order: 2016-09-04 21:15:19
Last order:  2018-10-17 17:30:18


order_purchase_timestamp
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

**Observations:**
- ~97% of orders are `delivered`; the analysis will focus on them.
- The data covers Sep 2016 – Oct 2018, but the first months (2016) and the last ones (Sep–Oct 2018)
  contain very few orders. These incomplete months will be excluded from trend charts.

## 7. Referential integrity

In [10]:
def orphans(child, child_key, parent, parent_key):
    return (~dfs[child][child_key].isin(dfs[parent][parent_key])).sum()

checks = {
    "order_items → orders": orphans("order_items", "order_id", "orders", "order_id"),
    "order_payments → orders": orphans("order_payments", "order_id", "orders", "order_id"),
    "order_reviews → orders": orphans("order_reviews", "order_id", "orders", "order_id"),
    "orders → customers": orphans("orders", "customer_id", "customers", "customer_id"),
    "order_items → products": orphans("order_items", "product_id", "products", "product_id"),
}
pd.Series(checks, name="rows without parent").to_frame()

,rows without parent
order_items → orders,0
order_payments → orders,0
order_reviews → orders,0
orders → customers,0
order_items → products,0


In [11]:
cats = dfs["products"]["product_category_name"].dropna().unique()
untranslated = set(cats) - set(dfs["category_translation"]["product_category_name"])
print("Categories without English name:", untranslated)

Categories without English name: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


## Summary

| Check | Result | Action in cleaning |
| --- | --- | --- |
| Dates stored as text | all timestamp columns | convert to `datetime` |
| Undelivered orders | ~3% | keep `delivered` only |
| Several reviews per order | small share | keep the latest review |
| Products without category | ~600 | fill with `unknown` |
| Untranslated categories | 2 | add English names manually |
| Repeat buyers | only via `customer_unique_id` | use it as customer key |
| Foreign keys | no orphan rows | safe to add FK constraints in PostgreSQL |

Next step → `02_cleaning.ipynb`.